# WaveletV3AE Successive-Halving Analysis

Loads per-dataset WaveletV3AE search results and cross-dataset benchmark results, then emits a markdown report.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.width', 180)
pd.set_option('display.max_colwidth', 80)

ROOT = Path('experiments')
if not ROOT.exists():
    ROOT = Path('.')

RESULTS = ROOT / 'results'
REPORT_PATH = ROOT / 'analysis_reports' / 'wavelet_v3ae_study_analysis.md'

DATASET_ORDER = ['m4', 'tourism', 'traffic', 'weather']
DATASET_LABELS = {
    'm4': 'M4-Yearly',
    'tourism': 'Tourism-Yearly',
    'traffic': 'Traffic-96',
    'weather': 'Weather-96',
}

SEARCH_CSVS = {
    ds: RESULTS / ds / 'wavelet_v3ae_study_results.csv'
    for ds in DATASET_ORDER
}
CROSS_CSV = RESULTS / 'wavelet_v3ae_cross_dataset_results.csv'


In [ ]:
def load_csv(path: Path):
    if not path.exists():
        return None
    df = pd.read_csv(path)
    for c in ['search_round', 'best_val_loss', 'trend_thetas_dim_cfg', 'latent_dim_cfg', 'basis_dim']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

search_dfs = {ds: load_csv(path) for ds, path in SEARCH_CSVS.items()}
cross_df = load_csv(CROSS_CSV)

for ds in DATASET_ORDER:
    df = search_dfs[ds]
    if df is None:
        print(f'{ds}: missing')
    else:
        print(f'{ds}: rows={len(df)} configs={df["config_name"].nunique()} rounds={sorted(df["search_round"].dropna().astype(int).unique().tolist())}')

print('cross rows:', 0 if cross_df is None else len(cross_df))

In [ ]:
report = []
report.append('# WaveletV3AE Study Analysis')
report.append('')

for ds in DATASET_ORDER:
    df = search_dfs[ds]
    report.append(f'## Dataset: {DATASET_LABELS[ds]}')
    report.append('')
    if df is None or df.empty:
        report.append(f'- Missing CSV: `{SEARCH_CSVS[ds]}`')
        report.append('')
        continue

    report.append(f'- CSV: `{SEARCH_CSVS[ds]}`')
    report.append(f'- Rows: {len(df)}')
    report.append(f'- Unique configs: {df["config_name"].nunique()}')
    report.append('')

    # Successive-halving funnel
    report.append('### Successive-Halving Funnel')
    rounds = sorted(df['search_round'].dropna().astype(int).unique().tolist())
    funnel_rows = []
    prev = None
    for r in rounds:
        rdf = df[df['search_round'].astype('Int64') == r]
        n_cfg = rdf['config_name'].nunique()
        keep = '-' if prev is None else f'{n_cfg}/{prev} ({n_cfg/max(prev,1):.0%})'
        best = pd.to_numeric(rdf['best_val_loss'], errors='coerce').median()
        funnel_rows.append({'Round': r, 'Configs': n_cfg, 'Rows': len(rdf), 'Median best_val_loss': best, 'Kept': keep})
        prev = n_cfg
    report.append(pd.DataFrame(funnel_rows).to_markdown(index=False, floatfmt='.4f'))
    report.append('')

    # Round 3 leaderboard
    report.append('### Round 3 Leaderboard (Top 10)')
    r3 = df[df['search_round'].astype('Int64') == 3].copy()
    if r3.empty:
        report.append('Round 3 not available.')
        report.append('')
    else:
        grp = (
            r3.groupby(['config_name', 'wavelet_family', 'bd_label', 'trend_thetas_dim_cfg', 'latent_dim_cfg'], dropna=False)
            .agg(median_best_val_loss=('best_val_loss', 'median'), std_best_val_loss=('best_val_loss', 'std'), n=('best_val_loss', 'count'))
            .sort_values('median_best_val_loss')
            .head(10)
            .reset_index()
        )
        report.append(grp.to_markdown(index=False, floatfmt='.4f'))
        report.append('')

    # Round 1 marginals
    report.append('### Round 1 Hyperparameter Marginals (median best_val_loss)')
    r1 = df[df['search_round'].astype('Int64') == 1].copy()
    for col in ['wavelet_family', 'bd_label', 'trend_thetas_dim_cfg', 'latent_dim_cfg']:
        if col not in r1.columns:
            continue
        mg = (
            r1.groupby(col)
            .agg(median_best_val_loss=('best_val_loss', 'median'), mean_best_val_loss=('best_val_loss', 'mean'), n=('best_val_loss', 'count'))
            .sort_values('median_best_val_loss')
            .reset_index()
        )
        report.append(f'#### {col}')
        report.append(mg.to_markdown(index=False, floatfmt='.4f'))
        report.append('')


In [ ]:
report.append('## Global Top-10 Selection Overview')
report.append('')

# Reconstruct global ranking inputs from round-3 dataset rankings
dataset_percentiles = {}
dataset_top10 = {}
for ds in DATASET_ORDER:
    df = search_dfs[ds]
    if df is None or df.empty:
        continue
    r3 = df[df['search_round'].astype('Int64') == 3].copy()
    if r3.empty:
        continue
    r3['cid'] = (
        r3['wavelet_family'].astype(str) + '|' +
        r3['bd_label'].astype(str) + '|ttd' +
        r3['trend_thetas_dim_cfg'].astype('Int64').astype(str) + '|ld' +
        r3['latent_dim_cfg'].astype('Int64').astype(str)
    )
    ranked = (
        r3.groupby('cid')
        .agg(median_best_val_loss=('best_val_loss', 'median'))
        .sort_values('median_best_val_loss')
        .reset_index()
    )
    n = len(ranked)
    denom = max(1, n - 1)
    dataset_percentiles[ds] = {row.cid: i / denom if n > 1 else 0.0 for i, row in ranked.iterrows()}
    dataset_top10[ds] = set(ranked.head(10)['cid'].tolist())

candidate_pool = set()
for s in dataset_top10.values():
    candidate_pool.update(s)

rows = []
for cid in sorted(candidate_pool):
    vals = []
    src = []
    for ds in DATASET_ORDER:
        p = dataset_percentiles.get(ds, {})
        vals.append(float(p.get(cid, 1.0)))
        if cid in dataset_top10.get(ds, set()):
            src.append(ds)
    rows.append({
        'canonical_config_id': cid,
        'mean_percentile': float(np.mean(vals)),
        'std_percentile': float(np.std(vals)),
        'source_datasets': ','.join(src),
    })

global_top10 = pd.DataFrame(rows).sort_values(['mean_percentile', 'std_percentile', 'canonical_config_id']).head(10) if rows else pd.DataFrame()
if global_top10.empty:
    report.append('No candidates found from round-3 dataset leaderboards.')
    report.append('')
else:
    report.append(global_top10.to_markdown(index=False, floatfmt='.4f'))
    report.append('')

report.append('## Cross-Dataset Robustness Leaderboard')
report.append('')
if cross_df is None or cross_df.empty:
    report.append(f'- Missing or empty cross CSV: `{CROSS_CSV}`')
    report.append('')
else:
    cdf = cross_df.copy()
    metric = 'best_val_loss'
    if metric not in cdf.columns and 'final_val_loss' in cdf.columns:
        metric = 'final_val_loss'
    board = (
        cdf.groupby(['canonical_config_id', 'source_datasets'], dropna=False)
        .agg(mean_metric=(metric, 'mean'), std_metric=(metric, 'std'), n=(metric, 'count'))
        .sort_values('mean_metric')
        .reset_index()
    )
    report.append(board.to_markdown(index=False, floatfmt='.4f'))
    report.append('')
    report.append(f'- Total cross rows: {len(cdf)}')
    report.append('- Expected full run count: 120 rows (10 configs x 4 datasets x 3 runs)')
    report.append('')

In [ ]:
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text('\n'.join(report), encoding='utf-8')
print(f'Wrote report to: {REPORT_PATH}')
print('\n'.join(report[:30]))